In [3]:
import sys
import os

# Get the absolute path to the sibling directory
sibling_dir = os.path.abspath('../01-agentic-rag')

# Add it to sys.path so Python knows to look there
if sibling_dir not in sys.path:
    sys.path.append(sibling_dir)

## Using sqlitesearch vector search in RAG

In this new notebook, set up the model and open the index (same as the "Reopening the index" section from `vector_search.ipynb` notebook):

In [1]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

We'll use the `RAGVector` class we defined in the previous lesson. It overrides the `search` method to embed the query and use vector search.

Set up the OpenAI client and create the assistant:

In [4]:
from rag_helper import RAGBase
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)

In [5]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes — you can still join even if the program has already begun. If you want a certificate, make sure to submit your project while submissions are still open.'

In [6]:
vs_index.close()